# Modelado de Inteligencia Artificial para la generación de caracteres estilo literario.

**Estudiantes:** Thiago Montenegro Yerumini.

---

### Importación de bibliotecas a usar para el modelado y análisis.

In [1]:
# Manejo de rutas locales.
from pathlib import Path

# Biblioteca principal para modelado.
import tensorflow as tf


### Introducción - Ingesta de datos.

#### Cargar el texto.
Acceder al texto limpio (sin formato, ni metadatos) y generar el vocabulario del mismo.

In [ ]:
# obtener ruta del directorio actual y construir la ruta la db (cuerpo de texto)
BASE_DIR = Path(__file__).resolve().parent
db_path = BASE_DIR / 'db' / 'The_Festival.txt'

text = db_path.read_text(encoding='utf-8')
print(f'Longitud del texto: {len(text)} caracteres.\n')

# obtenemos ocurrencias únicas de cada caracter en el texto (crear set) y lo ordenamos (sorted)
vocabulary = sorted(set(text))
print(f"Vocabulario ordenado: {vocabulary} \n {len(set(vocabulary))} caracteres únicos.")

NameError: name '__file__' is not defined

#### Tokenización y split de datos.
*Tokens*, dividir todo el texto en tokens por caracter para facilitar el procesamiento del modelo con la base de texto continuo.

*Diccionarios de búsqueda:* 
   1. Convertir cada caracter del vocabulario en un ID numérico.
   2. Obtener el caracter al que pertenece cada ID.

In [ ]:
# capas/funciones transformadoras: asignar a cada caracter su id
ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocabulary), mask_token=None)
# encontrar el caracter asigando para el id X
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(),
    invert=True,
    mask_token=None
)

# dividir todo el texto en caracteres individuales
all_tokens = tf.strings.unicode_split(text, 'UTF-8')

all_ids = ids_from_chars(all_tokens)

# parsear claves {id, char}, con reduce unimos los caracteres sueltos para formar la palabra
def text_from_ids(ids):
    return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

# total de tokens del texto ya vectorizado
n_tokens = len(all_ids)

# SPLIT (80% Train, 10% Dev, 10% Test), definir indices de corte 
train_end = int(n_tokens * 0.80)
dev_end = int(n_tokens * 0.90)

# partición contigua del tensor
train_ids = all_ids[:train_end]
dev_ids = all_ids[train_end:dev_end]
test_ids = all_ids[dev_end:]

print(f"Tokens Train: {len(train_ids)}")
print(f"Tokens Dev:   {len(dev_ids)}")
print(f"Tokens Test:  {len(test_ids)}")

### Dataset.
A partir del split de tokens para cada conjunto de datos, creamos dataset estructurados en base al tensor de ids.

In [ ]:
seq_length = 100
BATCH_SIZE = 64
BUFFER_SIZE = 10000

def create_dataset(ids_tensor, shuffle=False):
    # Crear flujo de datos desde los IDs
    ds = tf.data.Dataset.from_tensor_slices(ids_tensor)
    
    # Agrupar en fragmentos de tamaño seq_length + 1
    sequences = ds.batch(seq_length + 1, drop_remainder=True)
    
    # Separar entrada (X) y etiqueta (y)
    # Ejemplo: 'Hola' -> input: 'Hol', target: 'ola'
    def split_input_target(sequence):
        return sequence[:-1], sequence[1:]
    
    dataset = sequences.map(split_input_target)
    
    if shuffle:
        dataset = dataset.shuffle(BUFFER_SIZE)
        
    return dataset.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# Generar los 3 datasets
train_dataset = create_dataset(train_ids, shuffle=True)
dev_dataset   = create_dataset(dev_ids, shuffle=False)
test_dataset  = create_dataset(test_ids, shuffle=False)